# 암종 분류 — seed 앙상블 제출 재현

`09_final_submission.ipynb` 는 seed 42 하나로 v002(LB 0.4818)를 재현한다. 이 노트북은
거기에 **seed 앙상블**과 **하이퍼파라미터 프리셋** 축을 더해, 같은 파이프라인으로 여러
구성을 낼 수 있게 한 것이다. 기본 설정은 EXP_041 제출본을 그대로 재현한다.

| 축 | 기본값 | 바꾸는 곳 |
|---|---|---|
| 피처 | `f16` 16블록 5,309열 | `CONFIG` |
| 모델 | XGBoost · CatBoost · RandomForest | `MODEL_WEIGHTS` |
| seed | 42 / 7 / 2024 | `SEEDS` |
| CatBoost 파라미터 | `cbopt10` (Optuna trial10) | `PRESETS` |
| 분할 | `fold_group5` | `CV` |
| 결합 | 0.45 / 0.45 / 0.10 + 클래스별 로짓 보정(교차적합) | `MODEL_WEIGHTS` |
| 후처리 | 짝 라벨 규칙 `min_mut=3` | `MIN_MUT` |

기본 설정으로 나오는 것:

```
submission_ens16_cbopt10_seed3.csv            OOF 0.5210 · LB 0.3812     규칙 없음
submission_ens16_cbopt10_seed3_pairrule_m3.csv OOF 0.5210 · LB 0.4725    규칙 있음
```

## 두 가지 실행 방식

- **`REUSE_CACHE = True`** (기본) — `artifacts/oof/`·`artifacts/test_predictions/` 에
  이미 있는 9개 예측을 읽어 결합부터 한다. 몇 초. 결합·보정·짝 규칙을 확인하는 용도다.
- **`REUSE_CACHE = False`** — 9회(모델 3 × seed 3) 전부 새로 학습한다. GPU 기준 약 35분.

둘의 결과는 같아야 한다. 아래 §7 이 기준 파일과 전 행을 대조한다.

## 결론부터 — 이 구성은 채택하지 않았다

`cbopt10` 은 OOF 를 0.5165 에서 0.5210 으로 올리지만 **LB 는 0.3896 에서 0.3812 로
떨어진다.** 짝 규칙을 얹은 뒤에도 0.4818 대 0.4725 로 같은 크기만큼 손해다. 튜닝 이득이
train 분포에 붙어 있다는 뜻이고, 그 판정은 EXP_039 와 EXP_041 에서 두 번 나왔다.

**재현할 수 있게 남겨 두는 것이지 쓰라는 뜻이 아니다.** 현재 최고 제출은
`09_final_submission.ipynb` 가 내는 v002 + 짝 규칙(LB 0.4818)이다.

## 규정 준수

- 인코더·스케일러·집계 통계는 전부 **fold 의 train 부분에서만** fit 한다.
- 짝 라벨 규칙은 `train.csv` 에서만 유도되고 적용에는 test 한 행이면 된다.
- 외부 데이터 없음.
- **DACON 업로드는 사람이 직접 한다.** 이 노트북은 로컬 csv 만 만든다.

## 0. 준비

In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# 저장소 루트 — 노트북이 code/notebooks/ 에 있다는 전제
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "process"
ARTIFACTS = ROOT / "artifacts"

# ---- 설정 -------------------------------------------------------------------
SEEDS = [42, 7, 2024]
CONFIG = "f16"                 # 구현된 피처 16블록 전부, 5,309열
CV = "sgkf"                    # fold_group5 — 같은 변이 프로파일을 한 fold 로 묶는다
TOPK = 500                     # 유전자 블록 chi2 상위 K

MODEL_WEIGHTS = {"xgb": 0.45, "catboost": 0.45, "rf": 0.10}
TAGS = {"xgb": "repo16", "catboost": "cbopt10", "rf": "repo16"}
PRESETS = {"catboost": "cbopt10"}   # train_gbdt.PARAM_PRESETS 의 이름

MIN_MUT = 3                    # 짝 규칙이 요구하는 최소 변이 수
REUSE_CACHE = True             # False 면 9회 새로 학습 (~35분)

# 대조할 기준 파일. 없으면 대조를 건너뛴다.
REFERENCE = ROOT.parent / "Models/pairrule_candidates/submission_ens16_cbopt10_seed3_pairrule_m3.csv"
# -----------------------------------------------------------------------------

MODELS = list(MODEL_WEIGHTS)
print(f"ROOT = {ROOT}")
for name in ("train.csv", "test.csv", "sample_submission.csv"):
    assert (RAW / name).exists(), f"{name} 이 data/raw/ 에 없다"
assert (PROC / "train_folds.parquet").exists(), "scripts/make_folds.py 를 먼저 돌린다"

# fold 분할이 캐시된 예측을 만들 때와 같은지 확인한다. 다시 만들면 fold 경계가 바뀌고,
# 그러면 예전 OOF 와 새 OOF 를 섞는 순간 교차적합 보정이 valid fold 를 보게 된다.
# 파일 이름도 행 수도 그대로라 점수만 조용히 좋아지는 사고라 여기서 막는다.
from cancer_hack.provenance import EXPECTED_FOLD_FINGERPRINT, check_fold_fingerprint

fingerprint = check_fold_fingerprint(PROC / "train_folds.parquet")
folds_meta = pd.read_parquet(PROC / "train_folds.parquet")

print(f"모델 {MODELS} × seed {SEEDS} = 소스 {len(MODELS) * len(SEEDS)}개")
print(f"실행 방식: {'캐시 재사용' if REUSE_CACHE else '전부 재학습'}")
print(f"fold 지문: {fingerprint}  (기대 {EXPECTED_FOLD_FINGERPRINT}) — 일치")
print(f"  {len(folds_meta)}행 · fold_group5 크기 "
      f"{folds_meta['fold_group5'].value_counts().sort_index().tolist()}")

ROOT = D:\Code\Final_Hachathon\code
모델 ['xgb', 'catboost', 'rf'] × seed [42, 7, 2024] = 소스 9개
실행 방식: 캐시 재사용
fold 지문: 997d89a20595cc23  (기대 997d89a20595cc23) — 일치
  6201행 · fold_group5 크기 [1241, 1240, 1241, 1239, 1240]


### 버전 확인

블렌딩할 OOF 를 다른 기계에서 뽑아 합칠 때 버전이 어긋나면, 각자의 CV 는 멀쩡해 보이는데
**합쳐 놓은 결과만 조용히 어긋난다.** 그래서 예측을 바꾸는 4종은 고정한다.

In [2]:
import catboost, lightgbm, sklearn, xgboost

PINNED = {"scikit-learn": ("1.9.0", sklearn.__version__),
          "xgboost": ("3.3.0", xgboost.__version__),
          "lightgbm": ("4.7.0", lightgbm.__version__),
          "catboost": ("1.2.10", catboost.__version__)}
for pkg, (want, got) in PINNED.items():
    mark = "OK " if want == got else "다름"
    print(f"  [{mark}] {pkg:14s} 기준 {want:8s} 현재 {got}")
if any(w != g for w, g in PINNED.values()):
    print("\n※ 버전이 다르면 아래 점수가 재현되지 않는다. requirements.txt 를 맞춘다.")

  [OK ] scikit-learn   기준 1.9.0    현재 1.9.0
  [OK ] xgboost        기준 3.3.0    현재 3.3.0
  [OK ] lightgbm       기준 4.7.0    현재 4.7.0
  [OK ] catboost       기준 1.2.10   현재 1.2.10


## 1. 하이퍼파라미터 — `cbopt10` 이 무엇인가

EXP_039 의 Optuna 탐색 trial 10 이다. 이 값이 저장소 어디에도 없고 셸 명령줄에만 있어서
제출본을 재현할 수 없었다. 학습 로그의 `model_params` 에서 되찾아
`train_gbdt.PARAM_PRESETS` 에 등록했고, `--params cbopt10` 또는 아래처럼 고른다.

In [3]:
from train_gbdt import CONFIGS, PARAM_PRESETS, MODEL_PARAMS

print("CatBoost 기본값:")
for k, v in MODEL_PARAMS["catboost"].items():
    print(f"  {k:16s} {v}")

entry = PARAM_PRESETS["cbopt10"]
print(f"\ncbopt10 프리셋 — {entry['desc']}")
for k, v in entry["params"].items():
    print(f"  {k:16s} {v}")

print(f"\n{CONFIG} 블록 {len(CONFIGS[CONFIG]['blocks'])}개:")
print(" ", ", ".join(CONFIGS[CONFIG]["blocks"]))

CatBoost 기본값:
  iterations       1000
  learning_rate    0.05
  depth            6

cbopt10 프리셋 — EXP_039 Optuna trial10 — OOF +0.016 / LB -0.0084. 기각됨, 재현 전용
  iterations       1600
  learning_rate    0.1002086028456688
  depth            6
  l2_leaf_reg      1.0629966259002686
  subsample        0.510491669178009
  rsm              1

f16 블록 16개:
  domain, rollup, enc3, gec, gtype, parsed19, burden8, aa9, sigtok, exacttok, ptok, comut, lsvd, lnmf, gmod, csig


## 2. 소스 9개 — 학습하거나, 캐시에서 찾거나

`run_config` 가 파일명(stem)을 설정에서 만들어 낸다. 학습하면 그 stem 을 돌려받고,
캐시를 쓰면 같은 규칙으로 글롭해 찾는다. **0개나 2개 이상 맞으면 멈춘다** — 조용히
다른 실험의 예측을 섞는 게 이 파이프라인에서 제일 위험하다.

In [4]:
CV_SLUG = {"skf": "skf5", "sgkf": "group5"}[CV]

def find_cached(model: str, seed: int) -> str:
    """캐시된 예측의 stem 을 찾는다. 정확히 하나여야 한다."""
    pattern = f"oof_{model}_{TAGS[model]}_{CONFIG}_{CV_SLUG}_k{TOPK}_*_s{seed}.csv"
    hits = sorted((ARTIFACTS / "oof").glob(pattern))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"{pattern} 에 맞는 파일이 {len(hits)}개다. "
            "REUSE_CACHE=False 로 두고 새로 학습하거나 패턴을 확인한다."
        )
    return hits[0].stem[len("oof_"):]

def train(model: str, seed: int) -> str:
    """train_gbdt.run_config 를 그대로 부른다 — CLI 와 같은 코드 경로다."""
    from train_gbdt import build_parser, run_config
    args = build_parser().parse_args([])   # 피처 축 30여 개를 기본값으로 받는다
    args.model = model
    args.topk = TOPK
    args.n_splits = 5
    args.seed = seed
    args.device = "auto"
    args.override = {}
    args.params_preset = PRESETS.get(model)
    args.dry_run = False
    args.submission = False
    args.tag = TAGS[model]
    args.gpu_ram_part = 0.4
    return run_config(DATA, config=CONFIG, cv=CV, args=args)["stem"]

DATA = None
if not REUSE_CACHE:
    from train_gbdt import Dataset
    t0 = time.perf_counter()
    DATA = Dataset(set(CONFIGS[CONFIG]["blocks"]), n_splits=5)
    print(f"Dataset 준비 {time.perf_counter() - t0:.0f}초\n")

STEMS: dict[tuple[str, int], str] = {}
for model in MODELS:
    for seed in SEEDS:
        t0 = time.perf_counter()
        STEMS[(model, seed)] = find_cached(model, seed) if REUSE_CACHE else train(model, seed)
        note = "캐시" if REUSE_CACHE else f"{time.perf_counter() - t0:.0f}초"
        print(f"  {model:9s} seed {seed:<5d} [{note}]")

  xgb       seed 42    [캐시]
  xgb       seed 7     [캐시]
  xgb       seed 2024  [캐시]
  catboost  seed 42    [캐시]
  catboost  seed 7     [캐시]
  catboost  seed 2024  [캐시]
  rf        seed 42    [캐시]
  rf        seed 7     [캐시]
  rf        seed 2024  [캐시]


## 3. 앙상블 — 9소스 고정 가중 + 로짓 보정

가중치는 모델별 `0.45 / 0.45 / 0.10` 을 seed 수로 나눈다. seed 평균과 모델 가중을
따로 두지 않고 한 번에 섞는 것이 제출본이 한 방식이다.

보정은 클래스별로 로짓에 상수를 더해 결정 경계를 옮긴다. **교차적합으로 한다** —
fold 를 뺀 나머지에서 바이어스를 찾고 그 fold 에만 적용한다. 전체 OOF 에 한 번에
맞추면 0.5210 이 0.5438 로 부풀고 그 값으로는 구성을 고를 수 없다.

In [5]:
from cancer_hack.calibration import MacroF1LogitBias
from cancer_hack.ensemble import crossfit_calibrated_blend, weighted_average
from cancer_hack.metrics import macro_f1, read_prediction_frame

SOURCES = [(m, s) for m in MODELS for s in SEEDS]
WEIGHTS = [MODEL_WEIGHTS[m] / len(SEEDS) for m, _ in SOURCES]

oof_frames = [read_prediction_frame(ARTIFACTS / "oof" / f"oof_{STEMS[k]}.csv") for k in SOURCES]
test_frames = [read_prediction_frame(ARTIFACTS / "test_predictions" / f"test_{STEMS[k]}.csv")
               for k in SOURCES]

classes = oof_frames[0][1]
assert all(cs == classes for _, cs in oof_frames + test_frames), "소스마다 클래스 구성이 다르다"

columns = [f"p_{c}" for c in classes]
base_ids = oof_frames[0][0]["ID"].astype(str).to_numpy()

def aligned(frame, ids):
    """ID 로 정렬해 확률 행렬을 낸다. 행 순서 가정에 기대지 않는다."""
    indexed = frame.set_index(frame["ID"].astype(str)).reindex(ids)
    assert not indexed[columns].isna().any().any(), "소스 간 ID 집합이 다르다"
    return indexed[columns].to_numpy(dtype=np.float64)

oof = [aligned(f, base_ids) for f, _ in oof_frames]
test_ids = test_frames[0][0]["ID"].astype(str).to_numpy()
test = [aligned(f, test_ids) for f, _ in test_frames]

y = oof_frames[0][0]["y_true"].to_numpy()
folds = pd.read_parquet(PROC / "train_folds.parquet")
fold_ids = (folds.set_index(folds["ID"].astype(str)).reindex(base_ids)["fold_group5"].to_numpy())

print("소스별 OOF macro F1")
for (model, seed), values in zip(SOURCES, oof):
    print(f"  {model:9s} seed {seed:<5d} {macro_f1(y, np.asarray(classes)[values.argmax(1)]):.4f}")

# scripts/calibrate_ensemble.py 가 부르는 것과 같은 함수다.
raw_blend, crossfit, _ = crossfit_calibrated_blend(oof, y, classes, fold_ids, WEIGHTS)

uniform = weighted_average(oof, [1 / len(oof)] * len(oof))
print(f"\n균등 가중        OOF = {macro_f1(y, np.asarray(classes)[uniform.argmax(1)]):.4f}   (기록 0.5196)")
print(f"고정 가중 raw    OOF = {macro_f1(y, np.asarray(classes)[raw_blend.argmax(1)]):.4f}   (기록 0.5179)")
OOF_SCORE = macro_f1(y, np.asarray(classes)[crossfit.argmax(1)])
print(f"고정 가중 + 보정 OOF = {OOF_SCORE:.4f}   ← 보고할 값 (기록 0.5210)")

# test 에는 정답이 없어 교차적합을 못 한다. 전체 OOF 로 맞춘 바이어스를 쓴다.
final_bias = MacroF1LogitBias().fit(raw_blend, y, classes)
test_proba = final_bias.predict_proba(weighted_average(test, WEIGHTS))

소스별 OOF macro F1
  xgb       seed 42    0.4922
  xgb       seed 7     0.4880
  xgb       seed 2024  0.4878
  catboost  seed 42    0.5089
  catboost  seed 7     0.5097
  catboost  seed 2024  0.5110
  rf        seed 42    0.4774
  rf        seed 7     0.4797
  rf        seed 2024  0.4687



균등 가중        OOF = 0.5196   (기록 0.5196)
고정 가중 raw    OOF = 0.5179   (기록 0.5179)
고정 가중 + 보정 OOF = 0.5210   ← 보고할 값 (기록 0.5210)


## 4. 1층 제출 파일 (짝 규칙 없음)

여기까지가 **LB 0.3812** 다. OOF 는 v002(0.5165)보다 0.0045 높은데 LB 는 0.0084 낮다.

In [6]:
sample = pd.read_csv(RAW / "sample_submission.csv")
base_submission = pd.DataFrame({"ID": test_ids,
                                "SUBCLASS": np.asarray(classes)[test_proba.argmax(1)]})
base_submission = sample[["ID"]].astype(str).merge(base_submission, on="ID", validate="one_to_one")

assert len(base_submission) == 2546 and base_submission["SUBCLASS"].notna().all()
assert (base_submission["ID"].to_numpy() == sample["ID"].astype(str).to_numpy()).all()

base_path = ARTIFACTS / "submissions" / "submission_nb10_base.csv"
base_path.parent.mkdir(parents=True, exist_ok=True)
base_submission.to_csv(base_path, index=False, encoding="UTF-8-sig")
print(f"1층 제출: {base_path}   (기록 LB 0.3812)")
print(base_submission["SUBCLASS"].value_counts().head(5).to_string())

1층 제출: D:\Code\Final_Hachathon\code\artifacts\submissions\submission_nb10_base.csv   (기록 LB 0.3812)
SUBCLASS
STES     694
KIPAN    276
COAD     212
BRCA     178
PRAD     109


## 5. 2층 — 짝 라벨 규칙

암 분류 체계가 겹친다.

```
KIPAN(신장암 전체) = KICH + KIRC + KIRP
GBMLGG            = GBM  + LGG
```

같은 환자가 두 줄로 들어가 있고 유전자 4,384열은 글자 하나까지 같으며 라벨만 다르다.
그 두 줄이 train 과 test 로 갈라졌다. 모델은 "train 에서 KIRC 였으니 KIRC" 라고 답하는데,
train 쪽이 이미 KIRC 를 쓰고 있으므로 **test 쪽은 반드시 나머지 하나다.**

`fold_group5` 는 같은 프로파일을 한 fold 로 묶으므로 이 상황 자체를 못 만든다. 그래서
**OOF 는 규칙 적용 전후가 같다.** 근거는 `docs/pair_rule.md`.

In [7]:
from cancer_hack.pair_rule import PAIR, build_pair_rule

rule = build_pair_rule(RAW / "train.csv", RAW / "test.csv", min_mut=MIN_MUT)
d = rule.diagnostics

print(f"train 중복 묶음(변이 {MIN_MUT}개 이상)")
print(f"  라벨이 같은 묶음 : {d['train_dup_groups_same_label']:>4d}   ← 0 이어야 한다")
print(f"  코호트 짝인 묶음 : {d['train_dup_groups_pair_label']:>4d}")
print()
print(f"{'라벨':8s} {'train 밖 고아':>12s} {'test 매칭':>10s}")
for label in ("KIRC", "LGG", "KIPAN", "GBMLGG"):
    orphan = d["train_orphans_by_label"].get(label, 0)
    matched = d["test_matched_by_train_label"].get(label, 0)
    flag = "  <-- 정확히 일치" if orphan == matched else ""
    print(f"{label:8s} {orphan:12d} {matched:10d}{flag}")
print()
print(f"짝 4종이 아닌 매칭: {d['test_matched_off_pair_labels'] or '없음'}")
print(f"규칙 대상: {d['n_flipped']}행  (test 2,546행의 {d['n_flipped'] / 2546:.1%})")

# 전제가 깨지면 여기서 멈춘다 — 로컬 CV 로 검증할 수 없는 규칙이라
# 통과 못 한 채 제출하면 LB 한 번을 태우기 전까지 아무도 모른다.
violations = rule.verify_premises()
assert violations == [], violations

train 중복 묶음(변이 3개 이상)
  라벨이 같은 묶음 :    0   ← 0 이어야 한다
  코호트 짝인 묶음 :  422

라벨         train 밖 고아    test 매칭
KIRC               57         57  <-- 정확히 일치
LGG                50         50  <-- 정확히 일치
KIPAN             233         59
GBMLGG            280         48

짝 4종이 아닌 매칭: 없음
규칙 대상: 214행  (test 2,546행의 8.4%)


In [8]:
before = base_submission["SUBCLASS"].to_numpy().copy()
final_submission = base_submission.copy()
final_submission["SUBCLASS"] = rule.relabel(final_submission["ID"], before)

changed = int((final_submission["SUBCLASS"].to_numpy() != before).sum())
copied = sum(1 for i, c in zip(final_submission["ID"], before)
             if i in rule.mapping and PAIR.get(c) == rule.mapping[i])
print(f"바뀐 행 {changed} · 바꾸기 전 train 라벨을 복사하고 있던 행 {copied} "
      f"({copied / max(changed, 1):.1%})")

diff = pd.DataFrame({"before": before, "after": final_submission["SUBCLASS"]})
print()
print(diff[diff.before != diff.after].groupby(["before", "after"]).size().to_string())

바뀐 행 214 · 바꾸기 전 train 라벨을 복사하고 있던 행 214 (100.0%)

before  after 
GBMLGG  LGG       48
KIPAN   KIRC      59
KIRC    KIPAN     57
LGG     GBMLGG    50


## 6. 최종 제출 파일

In [9]:
final_path = ARTIFACTS / "submissions" / "submission_nb10_pairrule.csv"
final_submission.to_csv(final_path, index=False, encoding="UTF-8-sig")

# 스키마 검증 — 제출 전 마지막 관문
assert list(final_submission.columns) == ["ID", "SUBCLASS"]
assert len(final_submission) == 2546
assert (final_submission["ID"].to_numpy() == sample["ID"].astype(str).to_numpy()).all()
assert final_submission["SUBCLASS"].notna().all()
assert set(final_submission["SUBCLASS"]) <= set(classes)

print(f"최종 제출: {final_path}")
print(f"  2,546행 · {final_submission['SUBCLASS'].nunique()}클래스 · 스키마 검증 통과")
print()
print(f"  1층 OOF macro F1 = {OOF_SCORE:.4f}   (LB 0.3812)")
print(f"  2층 짝 규칙 {changed}행 교체   (LB 0.4725, +0.0913)")
print()
print("  ※ 짝 규칙은 OOF 를 바꾸지 않는다 — 같은 0.5210 에서 LB 만 움직인다.")
print("  ※ DACON 업로드는 사람이 직접 한다.")

최종 제출: D:\Code\Final_Hachathon\code\artifacts\submissions\submission_nb10_pairrule.csv
  2,546행 · 26클래스 · 스키마 검증 통과

  1층 OOF macro F1 = 0.5210   (LB 0.3812)
  2층 짝 규칙 214행 교체   (LB 0.4725, +0.0913)

  ※ 짝 규칙은 OOF 를 바꾸지 않는다 — 같은 0.5210 에서 LB 만 움직인다.
  ※ DACON 업로드는 사람이 직접 한다.


## 7. 기준 파일과 대조

실제로 제출해 LB 0.4725 를 받은 파일과 전 행을 맞춰 본다. 한 행이라도 다르면 이 노트북은
제출본을 재현하지 못하는 것이고, 그 상태로 코드를 내면 심사에서 코드와 제출물이 어긋난 것으로
보인다.

In [10]:
if REFERENCE.exists():
    reference = pd.read_csv(REFERENCE)
    same_ids = (reference["ID"].astype(str).to_numpy()
                == final_submission["ID"].astype(str).to_numpy()).all()
    n_diff = int((reference["SUBCLASS"].to_numpy()
                  != final_submission["SUBCLASS"].to_numpy()).sum())
    print(f"기준: {REFERENCE.name}")
    print(f"  ID 순서 일치 : {same_ids}")
    print(f"  다른 행      : {n_diff} / {len(reference)}")
    assert same_ids and n_diff == 0, "제출본을 재현하지 못했다"
    print("\n  재현 확인 — 2,546행 전부 일치")
else:
    print(f"기준 파일이 없어 대조를 건너뛴다: {REFERENCE}")

기준: submission_ens16_cbopt10_seed3_pairrule_m3.csv
  ID 순서 일치 : True
  다른 행      : 0 / 2546

  재현 확인 — 2,546행 전부 일치


## 8. 이 구성을 채택하지 않은 이유

| 베이스 | OOF | LB 무규칙 | LB 규칙 | 규칙 델타 |
|---|---|---|---|---|
| v002 f16 (seed 42, 기본 파라미터) | 0.5165 | 0.3896 | **0.4818** | +0.0922 |
| 이 노트북 (cbopt10 · 3-seed) | 0.5210 | 0.3812 | 0.4725 | +0.0913 |
| 차이 | +0.0045 | −0.0084 | −0.0093 | −0.0009 |

셋을 읽는다.

**1. 짝 규칙은 베이스와 거의 무관한 상수 가산이다.** 서로 다른 구성에서 +0.0922 와
+0.0913, 차이가 0.0009 다. 214행이 어느 클래스로 가는지가 베이스와 무관하게 정해져
있으니 당연하기도 하다.

**2. CatBoost 튜닝 + seed 앙상블은 규칙을 얹어도 여전히 손해다.** 무규칙 −0.0084,
규칙 −0.0093 으로 크기까지 비슷하다. 규칙이 베이스의 약점을 가려 주지 않는다.

**3. OOF 가 또 방향을 반대로 가리켰다.** 이 대회에서 OOF 가 오르고 LB 가 떨어진 경우가
이걸로 다섯 번째다.

seed 앙상블 자체는 작지만 재현되는 이득이다. 적합 파라미터가 0인 균등 블렌드에서
f16 1-seed 0.5138 → 3-seed 0.5160 으로 +0.0022, gec 정규화와 합쳐 +0.0040 이 나온다.
문제는 같이 들어간 `cbopt10` 쪽이다.

**최고 제출은 여전히 `09_final_submission.ipynb` 의 v002 + 짝 규칙(LB 0.4818)이다.**